# Script for generation of files and folder structure needed by NanoDiP

Aim: loop with curl commands to derive EpiDiP tumour classification for 16 cases


## Dependencies

In [ ]:
import openpyxl #has to be imported before pandas
from openpyxl import load_workbook
import pandas as pd
import subprocess
import os #added 2025-04-08
import urllib.request
import urllib.parse
from urllib.request import urlopen
import numpy as np
from urllib.request import urlopen
import requests
from pathlib import Path # added 2025-10-10 for creation of directories
import shutil #added 2025_10_17
from IPython.display import display

## Read in expanded MSA SampleSheet from Bonn for round 1 and 2
samples sheet specifes the putative tumour samples and their SentrixIDs- no cnv neutral samples

In [ ]:
def ReadMSASamplesheet():
    
    MSA_SampleSheetDF = pd.read_excel("/data/2026_01_15_IfP_MSA_idats/IfP_MSA_series_A.xlsx",sheet_name = 'MSA_samplesheet_round1_2', dtype={'SentrixBarcode_A': str, 'SentrixPosition_A': str}, header=8)

    #remove tailing .0 from SeSentrixBarcode_A
    MSA_SampleSheetDF['SentrixBarcode_A'] = MSA_SampleSheetDF['SentrixBarcode_A'].astype(str).str.replace('.0', '', regex=False)
    
    print('MSA_SampleSheetDF:')
    display(MSA_SampleSheetDF)
    
       
    MSA_SampleSheetDF['Sample_Name'] = MSA_SampleSheetDF['Sample_Name'].str.replace('.', '_', regex=False)
    MSA_SampleSheetDF['SentrixBarcode_A'] = MSA_SampleSheetDF['SentrixBarcode_A'].astype(str)
    MSA_SampleSheetDF['SentrixID'] = MSA_SampleSheetDF['SentrixBarcode_A']+"_"+ MSA_SampleSheetDF['SentrixPosition_A']

    print("MSA SampleSheetDF:")
    display(MSA_SampleSheetDF)
    
    return MSA_SampleSheetDF

## GenerateBetaThreshholdList
 AIM: create a list of Betavalues as Theshhold for Binaristaion of MSA betaValues

In [ ]:
def GenerateBetaThreshholdList(BetaStarti,BetaSTOPi):
    #AIM: create a list of Betavalues as Theshhold for Binaristaion of MSA betaValues
    Start=int(round(100*BetaStarti))
    Stop=int(round(100*BetaSTOPi))+1
    scaled_integers = range(Start, Stop)

# 2. Divide each integer by 100.0 to create the float with two decimals
    BetaThresholdList = [i / 100.0 for i in scaled_integers]

    return BetaThresholdList

## Working_MSAsampleSheet
for each betavalue a different string for a unique samlep name - inpupt amount is needed

In [ ]:
def Working_MSAsampleSheet(SampleSheetDF, BetaThrsholdListi, DateStampi):

    #create empty dataframe
    DF_Colheaders = ['Sample_Name','Input Menge ng','SentrixID']
    DF=pd.DataFrame(columns=DF_Colheaders)
        
    MSA_df_short = SampleSheetDF [['Sample_Name','Input Menge ng','SentrixID']]
        
    for BT in BetaThrsholdListi:
        print("BT:")
        display(BT)
        
                
        BT_Working_MSAsampleSheetDF = MSA_df_short
        BT_Working_MSAsampleSheetDF['BT'] = BT
        
        BT_str = (f"{BT:.2f}").replace('.', 'd')#convert BT float in string with 2 digit, peplace "." by "d"
        BT_Working_MSAsampleSheetDF['BT_str'] = BT_str
      
      
        DF=DF.append(BT_Working_MSAsampleSheetDF, ignore_index=True)

      #create in steps unique string representing uniqe Sample Name , DNA Input and BetaThreshold Value
    DF['InputStr']=(DF['Input Menge ng'].astype(str)).str.replace('\\.', 'p', regex=True)   
    DF['Sample_Name_InputM_Beta'] = DF['Sample_Name'] + "_" + DF['InputStr'] +  "_" + DF['BT_str']
    
    print("DF:")
    display(DF)    
    
    return (DF)

# GenerateMethoverlapCSV


In [ ]:
def GenerateMethoverlapCSV(ThisSentrixIDi, BTi):
    
    column_names=['ilmnID']
    MSA_probe_Names = pd.read_csv("/data/2026_01_15_IfP_MSA_idats/CPGs_208331750001_R01C02.txt", header=None, names=column_names)
    
    print("MSA_probe_Names:")
    display(MSA_probe_Names)
    
    betasFile="/data/SeSaMe_output/" + ThisSentrixIDi + "_sesame_19_1_10_prep_CDB_coll_PFX_T.bin"

    allBetaSingleFile = np.fromfile(betasFile, dtype=np.float64)
    
    #stepwise creation of Beta Dataframe     
    
    # 1)index is HM450/ Epic cg index    
    BetaDF=pd.read_csv('/applications/reference_data/ND_IfP_20250912_bin/index.csv', header=None, names=column_names)
    
    # 2) append vertically numpyarray with betavalues form bin file
    BetaDF['Beta'] = allBetaSingleFile
  
        
    #subset BetaDF for those cgs found on MSA array presented here in MSA_probe_Names
    subset_mask = BetaDF['ilmnID'].isin(MSA_probe_Names['ilmnID'])
    Beta_MSA_DF = BetaDF[subset_mask].reset_index(drop=True)
      
    stdDevDF = pd.read_csv("/data/epidip_temp/ND_IfP_20250912.xlsx_stdSortArray.csv")
    
    mergedDF = pd.merge(
                Beta_MSA_DF,
                stdDevDF,
                on='ilmnID',   
                how='left' 
                )
    
    
    SortedMergedDF = mergedDF.sort_values(by='StDev', ascending=False).reset_index()# mir unklar ob ich noch eine reset index brauche
   
    subsettedDF1 = SortedMergedDF.iloc[0:5000]# .loc['ilmnID', 'Beta']
    
    subsettedDF2=subsettedDF1[['ilmnID', 'Beta']]
  
    subsettedDF2['BinBeta']=(subsettedDF2['Beta'] >= np.float64(BTi)).astype(np.int8)
    
    subsettedDF3 = subsettedDF2[['ilmnID', 'BinBeta']]
    
    return(subsettedDF3)

# GenerateFolderFileStructure

In [ ]:
def GenerateFolderFileStructure(DateStamp,Working_MSAsampleetDFi):
      
    MSA_Datestamp_data_path = '/data' 
      
    i=0
    
    for row in Working_MSAsampleetDFi.itertuples(index=True, name="row"):
        
        s=row.Sample_Name_InputM_Beta
        
        sPATH=MSA_Datestamp_data_path + "/" + s + "/" + s
        sPathObject=Path(sPATH)
        sPathObject.mkdir(parents=True, exist_ok=True)        
        
        MethoverlapTsvFolderPath = MSA_Datestamp_data_path + "/nanodip_output/" + s + "/run1"
        MethoverlapTsvFolderPathObject=Path(MethoverlapTsvFolderPath)
        MethoverlapTsvFolderPathObject.mkdir(parents=True, exist_ok=True)
        
        MethoverlapTsvPath = MethoverlapTsvFolderPath +"/" + s+ "-methoverlap.tsv"
       
        ThisSentrixID=row.SentrixID
        
        BT = row.BT
        print('BT:')
        display(BT)
        print('input amount s:')
        display(s)
        
        #calculate Methoverlap.tsvfor this combination of BT, input and sample_Name in separate function
        MethO = GenerateMethoverlapCSV(ThisSentrixID,BT)
        MethO.to_csv(MethoverlapTsvPath,
                    sep='\t',      
                    header=False,  
                    index=False    # Exclude row index
                    )
        

        CpgCounttxtPath = MSA_Datestamp_data_path +"/nanodip_reports/" + s + "_cpgcount.txt"

        
        with open(CpgCounttxtPath, 'w') as file:
            # Write the string content to the file
            file.write(f"{5000}")
        
        #create aligned reads.txt for each "Sample"
        AlignedReadsTExtPath = MSA_Datestamp_data_path +"/nanodip_reports/" + s + "_alignedreads.txt"
        with open(AlignedReadsTExtPath, 'w') as ALRtxt:
            # Write the string content to the file
            ALRtxt.write(f"{0}")
        
        
        #copy cnv plot created by SeSaMe to the reports directory
        SourceDirectory = "/data/SeSaMe_output/"
        SourceFileName = ThisSentrixID + "_sesame_19_1_10_CNVplot.png"
        SourcePath = os.path.join(SourceDirectory, SourceFileName)
        
        
        DesinationDirctory = MSA_Datestamp_data_path + "/nanodip_reports/" 
        DestinationFilename = s + "_CNVplot.png"
        DestinationPath = os.path.join(DesinationDirctory, DestinationFilename)
        
        #copy the renamed file
        shutil.copy(SourcePath,DestinationPath) 
        
        
        
        i=i+1
    Working_MSAsampleetDF_path = MSA_Datestamp_data_path + "/" + "Working_MSAsampleetDF.csv"
    
    print('Working_MSAsampleetDF_path:')
    display(Working_MSAsampleetDF_path)
    
    Working_MSAsampleetDFi.to_csv(Working_MSAsampleetDF_path,
              sep=',',      # Use a tab character for TSV
              header=True,  # Exclude column names
              index=False    # Exclude row index
              )
    

# main

In [ ]:
if __name__ == '__main__':
    DateStamp="20260126"
    MSAsampleSheetDFex = ReadMSASamplesheet()
    BetaThresholdListEx=GenerateBetaThreshholdList(0.20,0.60)
    Working_MSAsamplSheetDFex = Working_MSAsampleSheet(MSAsampleSheetDFex, BetaThresholdListEx, DateStamp)
    GenerateFolderFileStructure(DateStamp,Working_MSAsamplSheetDFex)